# Census OCR Extraction — Proof of Concept
### Central Texas Spatial Demography Database Project (UT Austin IRP)

This notebook runs the full POC pipeline in Google Colab:
1. Upload one census scan image + the matching ground-truth XLSX sheet
2. Extract structured records from the image using Claude's vision API
3. Compare extracted records against the human-cleaned ground truth
4. Show accuracy metrics and a chart

**Verified starting point:** the 1950 Bastrop 11-2A sheet is confirmed to match
the sample images from this project (Line 1 = "Lewis Jasper H"), and has no known
data-entry typos — unlike some other 1950 sheets. Start here.

See `CLAUDE.md`, `PROMPTS.md`, and `IMPLEMENTATION.md` in the project repo for full context.

## 1. Setup

In [ ]:
!pip install -q anthropic openpyxl pandas fuzzywuzzy python-levenshtein opencv-python-headless

In [ ]:
import getpass, os

# Paste your Anthropic API key when prompted (input is hidden)
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

## 2. Upload your files

Upload:
- One census scan image (e.g. `43290879-Texas-112079-0002.jpg`)
- The matching ground-truth workbook (e.g. `Bastrop County 1950 Clean.xlsx`)

You can also mount Google Drive instead if your files live there (see the
commented-out cell below).

In [ ]:
from google.colab import files

print("Upload the census scan image...")
uploaded_image = files.upload()
image_path = list(uploaded_image.keys())[0]
print(f"Image saved as: {image_path}")

In [ ]:
print("Upload the ground-truth XLSX workbook...")
uploaded_gt = files.upload()
gt_path = list(uploaded_gt.keys())[0]
print(f"Ground truth saved as: {gt_path}")

In [ ]:
# Alternative: mount Google Drive instead of uploading each time
# from google.colab import drive
# drive.mount('/content/drive')
# image_path = "/content/drive/MyDrive/CTX_Census/raw_images/43290879-Texas-112079-0002.jpg"
# gt_path = "/content/drive/MyDrive/CTX_Census/ground_truth/Bastrop County 1950 Clean.xlsx" 

## 3. Configure the run

Set the census year and the exact ground-truth sheet name to compare against.

**Verified-clean 1950 sheets (no known typos):** `Bastrop 11-3`, `Bastrop 11-2A`,
`Smithville 11-10`, `Elgin 11-20`, `Elgin 11-21`.
Avoid `Bastrop all Manipulated`, `Bastrop 11-2B`, `Bastrop 11-1`,
`Smithville 11-7`, `Smithville 11-8`, `Smithville 11-9` for now — these contain
stray/typo values in their Race column that are ground-truth data-entry
mistakes, not extraction errors.

In [ ]:
CENSUS_YEAR = 1950
GROUND_TRUTH_SHEET = "Bastrop 11-2A"   # change to match your image

import pandas as pd
xl = pd.ExcelFile(gt_path)
print("Available sheets in this workbook:")
for s in xl.sheet_names:
    print(f"  - {s}")

## 4. Schema & normalization utilities

Verified directly against the ground truth files -- see `CLAUDE.md` for the full investigation. **Do not treat these as textbook census categories** — several are a research-team-specific controlled vocabulary that a vision model reading the raw image cannot produce directly (e.g. "Negro (Black)", "Mexican (Latino)", and especially the 1950 "W0"/"WO" sub-code, which is not visible anywhere on the physical form).

In [ ]:
BIRTHPLACE_COLUMN = {
    1850: "Birth Place", 1860: "Birth Place", 1950: "Birth Place",
    1870: "Birthplace", 1880: "Birthplace", 1900: "Birthplace",
    1910: "Birthplace", 1920: "Birthplace", 1930: "Birthplace", 1940: "Birthplace",
}

GENDER_COLUMN = {1920: "Sex"}  # every other decade uses "Gender"

HAS_LINE_NUMBER = {
    1850: True, 1860: False,  # 1860 confirmed to have NO Line Number column anywhere
    1870: True, 1880: True, 1900: True, 1910: True,
    1920: True, 1930: True, 1940: True, 1950: True,
}

# Race values ACTUALLY OBSERVED in the ground truth per decade (verified by
# scanning every sheet in every workbook -- not textbook categories)
VALID_RACE = {
    1850: ["White", "Black", "Mulatto"],
    1860: ["White", "Black", "Mulatto", "Indian (Native American)"],
    1870: ["White", "Black", "Mulatto"],
    1880: ["White", "Black", "Mulatto", "Filipino"],
    1900: ["White", "Black", "Mulatto", "Chinese", "Mexican (Latino)"],
    1910: ["White", "Black", "Mulatto", "Octoroon", "Mexican (Latino)", "Other"],
    1920: ["White", "Black", "Mulatto", "Mexican (Latino)"],
    1930: ["White", "Black", "Negro (Black)", "Mulatto", "Mexican (Latino)"],
    1940: ["White", "Negro (Black)"],
    1950: ["White", "Negro (Black)", "Chinese", "W0", "WO"],
}

# Raw form code (what a vision model reads off the image) -> ground truth string
RACE_NORMALIZE_MAP = {
    1950: {"W": "White", "Neg": "Negro (Black)", "Ch": "Chinese"},
    1940: {"W": "White", "Ne": "Negro (Black)", "Neg": "Negro (Black)"},
    1930: {"W": "White", "B": "Black", "Mu": "Mulatto", "Mex": "Mexican (Latino)", "Neg": "Negro (Black)"},
    1920: {"W": "White", "B": "Black", "Mu": "Mulatto", "Mex": "Mexican (Latino)"},
    1910: {"W": "White", "B": "Black", "Mu": "Mulatto", "Ot": "Octoroon", "Mex": "Mexican (Latino)"},
    1900: {"W": "White", "B": "Black", "Mu": "Mulatto", "Ch": "Chinese", "Mex": "Mexican (Latino)"},
    1880: {"W": "White", "B": "Black", "Mu": "Mulatto", "Fil": "Filipino"},
    1870: {"W": "White", "B": "Black", "Mu": "Mulatto"},
    1860: {"W": "White", "B": "Black", "Mu": "Mulatto", "In": "Indian (Native American)"},
    1850: {"W": "White", "B": "Black", "Mu": "Mulatto"},
    # "W0"/"WO" (1950) intentionally excluded -- not derivable from the image, see CLAUDE.md
}

VALID_GENDER = ["Male", "Female"]

VALID_MARITAL_STATUS = {
    1880: ["Married", "Single", "Widowed", "Widower", "Divorced", "Na"],
    1900: ["Married", "Single", "Widowed", "Divorced"],
    1910: ["Married", "Single", "Widowed", "Divorced"],
    1920: ["Married", "Single", "Widowed", "Divorced"],
    1930: ["Married", "Single", "Widowed", "Divorced"],
    1940: ["Married", "Single", "Widowed", "Divorced"],
    1950: ["Married", "Never Married (Single)", "Widowed", "Divorced", "Separated"],
}


def normalize_race(raw, year):
    if not raw:
        return raw
    raw = raw.strip()
    decade_map = RACE_NORMALIZE_MAP.get(year, {})
    if raw in decade_map:
        return decade_map[raw]
    for v in VALID_RACE.get(year, []):
        if raw.lower() == v.lower():
            return v
    return raw


def normalize_gender(raw):
    if not raw:
        return raw
    raw = raw.strip().upper()
    if raw in ["M", "MALE"]:
        return "Male"
    if raw in ["F", "FEMALE"]:
        return "Female"
    return raw


def propagate_dittos(records, surname_field="Surname", birthplace_field="Birthplace"):
    prev_surname, prev_birthplace = None, None
    ditto_markers = [None, "", "\u2014\u2014", '"', "''", "ditto", "do", "Do", "DO"]
    for rec in records:
        surname = rec.get(surname_field)
        if surname in ditto_markers:
            if prev_surname:
                rec[surname_field] = prev_surname
        else:
            prev_surname = surname
        birthplace = rec.get(birthplace_field)
        if birthplace in ditto_markers:
            if prev_birthplace:
                rec[birthplace_field] = prev_birthplace
        else:
            prev_birthplace = birthplace
    return records

print("Utilities loaded.")

# Maps the raw abbreviation a vision model reads off the form (Mar, Wd, D, S...)
# to the ground-truth string for that decade. CONFIRMED NECESSARY: fuzzy string
# matching fails badly on these (e.g. "mar" vs "married" scores only 60/100,
# "d" vs "divorced" scores 22/100, both well below any reasonable threshold) --
# without this normalization, correct extractions get marked as wrong.
MARITAL_STATUS_NORMALIZE_MAP = {
    1950: {"Mar": "Married", "Wd": "Widowed", "D": "Divorced", "Sep": "Separated",
           "S": "Never Married (Single)"},
    "default": {"Mar": "Married", "M": "Married", "Wd": "Widowed", "W": "Widowed",
                "D": "Divorced", "S": "Single", "Sep": "Separated"},
}


def normalize_marital_status(raw, year):
    if not raw:
        return raw
    raw = raw.strip()
    decade_map = MARITAL_STATUS_NORMALIZE_MAP.get(year, MARITAL_STATUS_NORMALIZE_MAP["default"])
    if raw in decade_map:
        return decade_map[raw]
    for v in VALID_MARITAL_STATUS.get(year, []):
        if raw.lower() == v.lower():
            return v
    return raw


## 5. Extraction prompt for this decade

Uses the verified schema and instructs the model to transcribe raw marks rather than expand or normalize them (normalization happens separately, below).

In [ ]:
SYSTEM_PROMPT = """You are a specialized historical census transcription assistant. You read
handwritten U.S. Census record images and extract structured data precisely.

RULES:
1. Transcribe EXACTLY what is written. Do not correct spelling of names.
2. If a field is blank or empty, use null.
3. If handwriting is truly illegible after careful inspection, use "[illegible]". Do NOT guess.
4. Ditto marks (\u2014\u2014, ", ditto) mean "same as the row above" -- write out the actual
   value, do not write the ditto mark itself.
5. "No one at home", "Vacant", or similar -- include as a record with line_number filled
   and all person fields null.
6. Return ONLY valid JSON. No explanation, no markdown fences, no preamble.
7. Race and marital status: transcribe the LITERAL mark on the page as written (e.g. "W"
   stays "W", "Neg" stays "Neg"). Do NOT expand abbreviations or map to any external
   category system -- normalization happens downstream, not here.
8. Gender/Sex: write "Male" or "Female" (safe to expand -- consistent across decades).
"""

PROMPTS_1950 = """This is a page from the 1950 U.S. Census of Population and Housing (Form P1).
State: Texas, County: Bastrop.

The form has two sections:
  - MAIN RECORDS (lines 1-30): extract all of these
  - SAMPLE LINES (bottom section, separate grid): label these with line_number as "S1", "S2", etc.

For each numbered line in MAIN RECORDS, extract using EXACTLY these key names:
  "Line Number"                integer
  "Street Name"                string or null
  "House Number"               integer or null
  "Dwelling Number"            integer or null
  "Surname"                    string or null (written once per household; propagate for ditto marks)
  "Given Name"                 string or null
  "Relation to Head of House"  string (Head, Wife, Son, Daughter, Lodger, etc.)
  "Race"                       string -- transcribe EXACTLY what letter/word is on the form (e.g. "W", "Neg")
  "Gender"                     string (Male or Female)
  "Age"                        integer or null
  "Marital Status"             string as abbreviated on form (Mar, Wd, D, Sep, S) or null
  "Birth Place"                string (state or country) or null
  "Occupation"                 string or null
  "Industry"                   string or null
  "Worker Class"               string (P, G, O, NP as marked) or null

Return a JSON array of objects, one per line.

NOTE: This decade's ground truth also uses a sub-code ("W0"/"WO") applied to some
White-coded individuals based on Spanish-language surnames. That code is NOT visible
anywhere on the physical form -- do not attempt to guess it. Just transcribe what's written.
"""

DECADE_PROMPTS = {1950: PROMPTS_1950}  # add other decades from PROMPTS.md as needed

if CENSUS_YEAR not in DECADE_PROMPTS:
    raise ValueError(
        f"No prompt defined in this notebook for {CENSUS_YEAR}. "
        f"Copy the matching block from PROMPTS.md in the project repo."
    )
print(f"Prompt ready for {CENSUS_YEAR}.")

## 6. Run extraction

In [ ]:
import anthropic, base64, json, re

def image_to_base64(path):
    with open(path, "rb") as f:
        data = f.read()
    media_type = "image/png" if path.lower().endswith(".png") else "image/jpeg"
    return base64.standard_b64encode(data).decode("utf-8"), media_type


def extract_from_image(image_path, year, client=None):
    if client is None:
        client = anthropic.Anthropic()
    b64_data, media_type = image_to_base64(image_path)
    user_prompt = DECADE_PROMPTS[year]

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4096,
        system=SYSTEM_PROMPT,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": b64_data}},
                {"type": "text", "text": user_prompt},
            ],
        }],
    )

    raw = response.content[0].text
    raw = re.sub(r"```json\s*", "", raw)
    raw = re.sub(r"```\s*$", "", raw, flags=re.MULTILINE).strip()
    records = json.loads(raw)

    records = propagate_dittos(records, surname_field="Surname",
                                birthplace_field=BIRTHPLACE_COLUMN.get(year, "Birthplace"))
    gender_field = GENDER_COLUMN.get(year, "Gender")
    for rec in records:
        if "Race" in rec:
            rec["Race"] = normalize_race(rec["Race"], year)
        if "Marital Status" in rec:
            rec["Marital Status"] = normalize_marital_status(rec["Marital Status"], year)
        if gender_field in rec:
            rec[gender_field] = normalize_gender(rec[gender_field])
        elif "Gender" in rec:
            rec["Gender"] = normalize_gender(rec["Gender"])

    return records


print("Extracting records from image... (this calls the Anthropic API)")
extracted_records = extract_from_image(image_path, CENSUS_YEAR)
print(f"Extracted {len(extracted_records)} records.")
extracted_records[:3]  # preview first few

## 7. Compare against ground truth

Field-matching strategy is chosen by keyword, not a fixed column list, since column names genuinely differ by decade (`"Birthplace"` vs `"Birth Place"`, `"Gender"` vs `"Sex"`, etc. -- verified, see CLAUDE.md).

**Important:** a ground-truth sheet like `Bastrop 11-2A` is not one physical census page -- it's dozens of physical pages concatenated together, with Line Number resetting to 1 at the start of each one (confirmed: 24 separate pages inside `Bastrop 11-2A` alone). You must tell `compare()` which physical page to check against via `PHYSICAL_PAGE` below, or you will silently get matched against the wrong data.

In [ ]:
from fuzzywuzzy import fuzz
import pandas as pd

STRATEGY_KEYWORDS = [
    (["race"], "exact"),
    (["gender", "sex"], "exact"),
    (["marital status"], "exact"),
    (["age"], "numeric"),
    (["surname", "given name", "first name", "last name"], "fuzzy_name"),
    (["relation"], "fuzzy"),
    (["birthplace", "birth place"], "fuzzy"),
    (["occupation", "industry"], "fuzzy"),
]

def classify_strategy(field_name):
    fl = field_name.lower()
    for keywords, strategy in STRATEGY_KEYWORDS:
        if any(k in fl for k in keywords):
            return strategy
    return "fuzzy"

def _norm(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return ""
    return str(val).strip().lower()

def field_match(ext_val, gt_val, strategy):
    e, g = _norm(ext_val), _norm(gt_val)
    if e == "" and g == "":
        return True
    if e == "" or g == "":
        return False
    if strategy == "exact":
        return e == g
    if strategy in ("fuzzy", "fuzzy_name"):
        threshold = 90 if strategy == "fuzzy_name" else 85
        return fuzz.ratio(e, g) >= threshold
    if strategy == "numeric":
        try:
            return abs(int(float(e)) - int(float(g))) <= 1
        except (ValueError, TypeError):
            return e == g
    return e == g


def find_line_number_column(columns):
    for c in columns:
        if c and str(c).strip().lower() == "line number":
            return c
    return None


def split_into_physical_pages(gt_df, line_col):
    """
    CRITICAL: a ground-truth sheet like "Bastrop 11-2A" is not one physical
    census page -- it's many physical pages concatenated end to end (confirmed:
    24 pages for Bastrop 11-2A alone), and Line Number resets to 1 at the start
    of each new physical page. This splits the sheet into page-blocks so you
    compare against the correct one instead of accidentally matching a random
    later page that happens to share the same line numbers 1-30.
    """
    line_numbers = pd.to_numeric(gt_df[line_col], errors="coerce")
    page_starts = [0]
    for i in range(1, len(line_numbers)):
        prev, curr = line_numbers.iloc[i - 1], line_numbers.iloc[i]
        if pd.notna(curr) and curr == 1 and (pd.isna(prev) or prev != 1):
            page_starts.append(i)
    page_starts.append(len(gt_df))
    return [gt_df.iloc[s:e].reset_index(drop=True) for s, e in zip(page_starts, page_starts[1:])]


def compare(records, gt_xlsx_path, gt_sheet, year, physical_page):
    """
    physical_page: 1-indexed physical page within gt_sheet, matching the
    "Sheet Number" printed/handwritten on your source scan image. Always
    cross-check against the scan itself -- don't assume page 1 = your image
    without verifying a name or two match.
    """
    if not HAS_LINE_NUMBER.get(year, True):
        raise ValueError(
            f"{year} sheets have no Line Number column -- row alignment needs "
            f"a different key for this decade, not implemented in this notebook."
        )

    gt_df = pd.read_excel(gt_xlsx_path, sheet_name=gt_sheet)
    line_col = find_line_number_column(gt_df.columns)
    if line_col is None:
        raise ValueError(f"No 'Line Number' column found in sheet '{gt_sheet}'.")

    pages = split_into_physical_pages(gt_df, line_col)
    print(f"Sheet '{gt_sheet}' contains {len(pages)} physical pages. Using page {physical_page}.")
    if not (1 <= physical_page <= len(pages)):
        raise ValueError(f"physical_page={physical_page} out of range (1..{len(pages)}).")
    page_df = pages[physical_page - 1]

    gt_records = page_df.to_dict(orient="records")
    ext_by_line = {int(r["Line Number"]): r for r in records if r.get("Line Number") is not None}
    gt_by_line = {int(r[line_col]): r for r in gt_records if r.get(line_col) is not None}

    compare_columns = [c for c in page_df.columns if c and c != line_col]
    results = []
    field_scores = {c: [] for c in compare_columns}

    for line_num in sorted(gt_by_line.keys()):
        gt_row = gt_by_line[line_num]
        ext_row = ext_by_line.get(line_num, {})
        row_result = {"line_number": line_num, "all_match": True, "fields": {}}

        for field in compare_columns:
            strategy = classify_strategy(field)
            gt_val = gt_row.get(field)
            ext_val = ext_row.get(field)
            matched = field_match(ext_val, gt_val, strategy)
            row_result["fields"][field] = {"extracted": ext_val, "ground_truth": gt_val, "match": matched}
            field_scores[field].append(matched)
            if not matched:
                row_result["all_match"] = False

        results.append(row_result)

    n = len(results)
    metrics = {
        "census_year": year,
        "sheet": gt_sheet,
        "physical_page": physical_page,
        "total_physical_pages_in_sheet": len(pages),
        "rows_compared": n,
        "row_accuracy": sum(r["all_match"] for r in results) / n if n else 0,
        "field_accuracy": {f: (sum(s) / len(s) if s else None) for f, s in field_scores.items()},
    }
    scored = {k: v for k, v in metrics["field_accuracy"].items() if v is not None}
    metrics["overall_field_accuracy"] = sum(scored.values()) / len(scored) if scored else 0
    return metrics, results


PHYSICAL_PAGE = 1  # change this to match which physical page your image shows
                    # (check the "Sheet Number" field on the scan itself)

metrics, results = compare(extracted_records, gt_path, GROUND_TRUTH_SHEET, CENSUS_YEAR, PHYSICAL_PAGE)

print(f"Rows compared:          {metrics['rows_compared']}")
print(f"Row-level accuracy:     {metrics['row_accuracy']:.1%}")
print(f"Overall field accuracy: {metrics['overall_field_accuracy']:.1%}")

## 8. Visualize field-level accuracy

In [ ]:
import matplotlib.pyplot as plt

scored_fields = {k: v for k, v in metrics["field_accuracy"].items() if v is not None}
sorted_fields = dict(sorted(scored_fields.items(), key=lambda x: x[1]))

fig, ax = plt.subplots(figsize=(9, max(4, len(sorted_fields) * 0.4)))
bars = ax.barh(list(sorted_fields.keys()), [v * 100 for v in sorted_fields.values()])
ax.set_xlabel("Accuracy (%)")
ax.set_xlim(0, 100)
ax.set_title(f"Field-Level Extraction Accuracy \u2014 {metrics['census_year']} Census, sheet '{metrics['sheet']}'")
ax.axvline(x=metrics["overall_field_accuracy"] * 100, color="red", linestyle="--",
           label=f"Overall: {metrics['overall_field_accuracy']:.1%}")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Inspect mismatches

Most useful step for deciding what to fix next: real extraction errors vs. known ground-truth limitations (like the 1950 W0/WO sub-code, or typos in certain sheets — see CLAUDE.md).

In [ ]:
mismatches = []
for r in results:
    for field, detail in r["fields"].items():
        if not detail["match"]:
            mismatches.append({
                "line": r["line_number"], "field": field,
                "extracted": detail["extracted"], "ground_truth": detail["ground_truth"],
            })

mismatch_df = pd.DataFrame(mismatches)
print(f"Total field mismatches: {len(mismatch_df)}")
mismatch_df.head(30)

## 10. Save results

In [ ]:
import json

with open("poc_results.json", "w") as f:
    json.dump({"metrics": metrics, "results": results}, f, indent=2)

from google.colab import files
files.download("poc_results.json")
print("Saved and downloading poc_results.json")

## Notes on interpreting your accuracy numbers

- **Race field accuracy will not reach 100% on 1950 data** even with a perfect
  extractor. The ground truth's "W0"/"WO" sub-code is applied by the research
  team based on surname, not written anywhere on the physical census form —
  see CLAUDE.md section "VERIFIED Data Quality Findings" for the full
  investigation. Don't chase this to zero; flag it as a known, documented gap
  when you present results to Jaden.
- If you compare against a sheet other than `Bastrop 11-2A`, `Bastrop 11-3`,
  `Smithville 11-10`, `Elgin 11-20`, or `Elgin 11-21`, you may see stray
  ground-truth values (`'Whiite'`, `'White0'`, `'1'`, `'72'`, `'S'`, etc.) that
  are typos in the "clean" file itself, not extraction errors.
- To compare against Ancestry's own baseline OCR accuracy (to show the delta
  your pipeline achieves), use the paired Raw/Clean sheets in
  `Raw-Clean Comparison Document.xlsx` the same way — swap `gt_path` for that
  file and pick a `*_Raw` vs `*_Clean` sheet pair.